# Recipe Review Sentiment Analysis
## A 3-Class NLP Classification Pipeline — Linear SVC with Enhanced Feature Engineering

### Problem Statement
This notebook builds a robust, production-ready 3-class sentiment classifier for recipe reviews.
Given a textual review and star rating (1–5), we map labels as:
- **Negative (0):** Stars 1–2
- **Neutral (1):** Star 3
- **Positive (2):** Stars 4–5

### Dataset
`Recipe Reviews and User Feedback Dataset.csv` — contains `text` (review body) and `stars` (1–5 rating).

### Methodology Overview
1. **EDA** — Class distribution, review length analysis, sample inspection, feature correlations
2. **Preprocessing** — HTML stripping, regex cleaning, lowercasing
3. **Feature Engineering** — Delta TF-IDF (1–3 grams), expanded VADER (pos/neu/neg/compound), TextBlob, POS counts, negation, lexical richness
4. **Baselines** — Majority-class and VADER-only rule-based classifiers
5. **Modelling** — Linear SVC (untuned → Optuna-tuned with 5-fold CV + expanded search space)
6. **Metrics** — Accuracy, Macro-F1, and per-class Precision / Recall / F1
7. **Error Analysis** — Inspection of misclassified samples with focus on Neutral class
8. **Serialization** — Tuned model and full preprocessing pipeline saved as `.pkl` files

### Reproducibility
All random seeds are fixed at `SEED = 42`, including Optuna samplers.


In [ ]:
# ─────────────────────────────────────────────
# Cell 1 | Package Installation
# ─────────────────────────────────────────────
import subprocess, sys

packages = [
    'pandas', 'numpy', 'scikit-learn', 'nltk', 'beautifulsoup4',
    'matplotlib', 'seaborn', 'wordcloud',
    'scipy', 'spacy', 'vaderSentiment', 'textblob', 'lightgbm', 'optuna'
]

print('Installing/verifying packages...')
for pkg in packages:
    import_name = {
        'scikit-learn': 'sklearn',
        'beautifulsoup4': 'bs4',
        'vaderSentiment': 'vaderSentiment'
    }.get(pkg, pkg.replace('-', '_'))
    try:
        __import__(import_name)
        print(f'  [OK] {pkg}')
    except ImportError:
        print(f'  Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('\nChecking spaCy model...')
try:
    import spacy; spacy.load('en_core_web_sm'); print('  [OK] en_core_web_sm')
except Exception:
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'])

import nltk
for resource in ['punkt', 'averaged_perceptron_tagger', 'stopwords',
                 'punkt_tab', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

print('\nAll dependencies ready.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 2 | Imports & Global Config
# ─────────────────────────────────────────────
import re, html, os, warnings
from typing import Tuple, Dict, List, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import hstack, csr_matrix

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import spacy
from bs4 import BeautifulSoup
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

import optuna
import joblib

# ── Global constants ──
SEED   = 42
LABELS = ['Neg', 'Neu', 'Pos']

# ── Global NLP singletons (initialised once) ──
analyzer = SentimentIntensityAnalyzer()
nlp      = spacy.load('en_core_web_sm', disable=['ner', 'parser'])

# ── Global results log (one row per model) ──
results_log: List[Dict] = []

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
np.random.seed(SEED)

print('All libraries imported. SEED =', SEED)

---
## Section 1 — Exploratory Data Analysis (EDA)

Before any modelling we inspect the raw data to understand:
- **Class balance** — are the three sentiment classes equally represented?
- **Review length** — do positive reviews tend to be shorter or longer than negative ones?
- **Sample inspection** — does the text look clean, are there HTML artefacts?
- **Feature correlations** — do VADER / TextBlob scores track with the gold labels?

In [ ]:
# ─────────────────────────────────────────────
# Cell 3 | Load Raw Data & Basic EDA
# ─────────────────────────────────────────────
CSV_PATH = 'Recipe Reviews and User Feedback Dataset.csv'

raw = pd.read_csv(CSV_PATH)
print('Shape:', raw.shape)
print('Columns:', raw.columns.tolist())
print('\nMissing values:\n', raw.isnull().sum())
print('\nStar-rating distribution:\n', raw['stars'].value_counts().sort_index())

In [ ]:
# ─────────────────────────────────────────────
# Cell 4 | EDA Plots
# ─────────────────────────────────────────────
eda_df = raw.dropna(subset=['text', 'stars']).copy()
eda_df['label'] = eda_df['stars'].map({1: 0, 2: 0, 3: 1, 4: 2, 5: 2})
eda_df['word_count']  = eda_df['text'].astype(str).str.split().str.len()
eda_df['vader_score'] = eda_df['text'].astype(str).apply(
    lambda x: analyzer.polarity_scores(x)['compound'])
eda_df['tb_score'] = eda_df['text'].astype(str).apply(
    lambda x: TextBlob(x).sentiment.polarity)

label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
eda_df['label_name'] = eda_df['label'].map(label_names)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Class distribution (count)
counts = eda_df['label_name'].value_counts().reindex(['Negative', 'Neutral', 'Positive'])
axes[0, 0].bar(counts.index, counts.values,
               color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white')
axes[0, 0].set_title('Class Distribution (Count)')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0, 0].text(i, v + 20, f'{v:,}', ha='center', fontsize=9)

# 2. Class distribution (percentage)
pcts = counts / counts.sum() * 100
axes[0, 1].pie(pcts.values, labels=pcts.index, autopct='%1.1f%%',
               colors=['#e74c3c', '#f39c12', '#2ecc71'], startangle=90)
axes[0, 1].set_title('Class Distribution (%)')

# 3. Review word count by class
for lname, color in [('Negative', '#e74c3c'), ('Neutral', '#f39c12'), ('Positive', '#2ecc71')]:
    subset = eda_df[eda_df['label_name'] == lname]['word_count'].clip(upper=300)
    axes[0, 2].hist(subset, bins=40, alpha=0.55, label=lname, color=color)
axes[0, 2].set_title('Review Word Count by Class')
axes[0, 2].set_xlabel('Word count (capped at 300)')
axes[0, 2].legend()

# 4. VADER score by class (boxplot)
order = ['Negative', 'Neutral', 'Positive']
eda_df.boxplot(column='vader_score', by='label_name', ax=axes[1, 0],
               positions=[0, 1, 2], patch_artist=True)
axes[1, 0].set_title('VADER Compound Score by Class')
axes[1, 0].set_xlabel('')
plt.sca(axes[1, 0]); plt.xticks([0, 1, 2], order)

# 5. TextBlob polarity by class (boxplot)
eda_df.boxplot(column='tb_score', by='label_name', ax=axes[1, 1],
               positions=[0, 1, 2], patch_artist=True)
axes[1, 1].set_title('TextBlob Polarity by Class')
axes[1, 1].set_xlabel('')
plt.sca(axes[1, 1]); plt.xticks([0, 1, 2], order)

# 6. VADER vs TextBlob scatter (2 000-point sample)
sample = eda_df.sample(min(2000, len(eda_df)), random_state=SEED)
for lbl, color in [(0, '#e74c3c'), (1, '#f39c12'), (2, '#2ecc71')]:
    sub = sample[sample['label'] == lbl]
    axes[1, 2].scatter(sub['vader_score'], sub['tb_score'],
                       alpha=0.3, s=10, color=color, label=label_names[lbl])
axes[1, 2].set_title('VADER vs TextBlob (sample)')
axes[1, 2].set_xlabel('VADER compound')
axes[1, 2].set_ylabel('TextBlob polarity')
axes[1, 2].legend(markerscale=3)

fig.suptitle('Exploratory Data Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eda_plots.png', bbox_inches='tight')
plt.show()
print('EDA plots saved.')

In [ ]:
# ─────────────────────────────────────────────
# Cell 5 | Sample Inspection & Summary Stats
# ─────────────────────────────────────────────
print('=== SAMPLE REVIEWS PER CLASS ===')
for lbl, name in label_names.items():
    print(f'\n--- {name} ---')
    samples = eda_df[eda_df['label'] == lbl]['text'].sample(3, random_state=SEED)
    for i, txt in enumerate(samples, 1):
        print(f'  [{i}] {str(txt)[:200].replace(chr(10), " ")}...')

print('\n=== SUMMARY STATISTICS ===')
print(eda_df.groupby('label_name')[['word_count', 'vader_score', 'tb_score']]
      .agg(['mean', 'median', 'std']).round(3))

---
## Section 2 — Preprocessing & Feature Engineering

### Key design decisions
- **No leakage**: `TfidfVectorizer`, delta weights, and `MaxAbsScaler` are all fit **only on the training split**. Val and test sets are transformed using the already-fit objects.
- **No SMOTE**: Class imbalance is handled entirely via `class_weight='balanced'` inside each classifier.
- **3-class Delta TF-IDF**: Each TF-IDF column is multiplied by a combined delta `(pos − neg) + 0.5 × (neu − mean(pos, neg))` computed from training data, biasing toward polarity-discriminative terms and giving the Neutral class its own discriminative signal.
- **Expanded VADER**: All four VADER scores (`pos`, `neu`, `neg`, `compound`) are used as features. The `neu` score is especially informative for the Neutral class.
- **Richer meta features**: 25 features total including negation density, exclamation/question mark counts, lexical richness (TTR), sentence count, TextBlob subjectivity, and both positive and negative keyword counts.
- **spaCy cap at 1000 chars**: Balances POS coverage vs. speed.
- **Scaler**: `MaxAbsScaler` is fit on training data and applied consistently to val/test.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 6 | Preprocessing Helpers
# ─────────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """Strip HTML entities and tags, remove non-alpha chars, lowercase.
    Used for TF-IDF and spaCy POS — NOT for sentiment scorers.
    """
    text = html.unescape(str(text))
    text = BeautifulSoup(text, 'html.parser').get_text()
    text = re.sub(r'[^a-zA-Z\s!?]', '', text)
    return text.lower().strip()


def make_vader_input(text: str) -> str:
    """Light-touch cleaning for VADER / TextBlob scoring.

    Only HTML-unescapes and strips HTML tags; casing, punctuation,
    contractions, exclamation marks, and numbers are all preserved so
    VADER's emphasis rules fire correctly (e.g. 'GREAT!!!' > 'great').
    This fixes the original bug where clean_text was used for VADER,
    stripping capitals and punctuation that VADER relies on.
    """
    text = html.unescape(str(text))
    return BeautifulSoup(text, 'html.parser').get_text()


# Negation trigger words — used for negation density feature
NEGATION_WORDS = {
    'not', "n't", 'no', 'never', 'neither', 'nor',
    'nothing', 'nobody', 'nowhere', 'hardly', 'barely', 'scarcely'
}

# Positive and negative keyword lexicons
POS_KEYWORDS = re.compile(
    r'excellent|great|good|perfect|amazing|fantastic|delicious|wonderful|'
    r'loved|love|brilliant|outstanding|superb|best|incredible|highly', re.I)
NEG_KEYWORDS = re.compile(
    r'terrible|awful|bad|horrible|disgusting|worst|disappointing|'
    r'bland|tasteless|inedible|waste|dry|soggy|overcooked|undercooked', re.I)

# Intent patterns — fixed: single \b word boundaries, apostrophe variants
NEGATIVE_INTENT_RE = re.compile(
    r"\b(won'?t|would\s+not|will\s+not|wouldn'?t)\s+(make|try|cook|use|do)\s+(again|this|it)\b"
    r"|\b(never\s+(making|trying|cooking)\s+again)\b"
    r"|\b(not\s+worth\s+(making|trying|it))\b",
    re.I,
)
POSITIVE_INTENT_RE = re.compile(
    r"\b(will|would|'ll)\s+(make|try|cook|use|do)\s+(again|this)\b"
    r"|\b(making\s+(this|it)\s+again)\b"
    r"|\b(definitely\s+(recommend|making|trying))\b",
    re.I,
)


def extract_meta_features(df: pd.DataFrame) -> np.ndarray:
    """
    Return an (N, 22) float array of rich meta features.

    Features
    --------
    0  vader_pos          — VADER positive score
    1  vader_neu          — VADER neutral score  ← key for Neutral class
    2  vader_neg          — VADER negative score
    3  vader_compound     — VADER compound score
    4  textblob_polarity
    5  textblob_subjectivity
    6  verb_count         — spaCy POS (cap 1000 chars)
    7  adj_count          — spaCy POS
    8  adv_count          — spaCy POS
    9  pos_keyword_count  — positive lexicon hits
    10 neg_keyword_count  — negative lexicon hits
    11 negation_density   — negation words / word count
    12 word_count
    13 exclamation_count
    14 question_count
    15 type_token_ratio   — lexical richness (TTR)
    16 sentence_count
    17 sentence_vader_std  — std-dev of per-sentence VADER compounds;
                            high value → "bipolar" mixed-sentiment review
    18 vader_contrast      — abs(vader_pos − vader_neg)
    19 last_sentence_vader — VADER compound of the final sentence
    20 negative_intent_flag — 1 if "won't make again" etc.
    21 positive_intent_flag — 1 if "will make again" etc.
    22 caps_ratio           — uppercase chars / total chars; signals ALL-CAPS anger/emphasis
    23 avg_sentence_length  — word_count / sentence_count; short choppy = negative signal
    24 first_sentence_vader — VADER compound of the opening sentence

    VADER/TextBlob scoring note: all sentiment scores use 'vader_input'
    (HTML-unescaped, tag-stripped, but casing/punctuation intact) so
    that VADER's emphasis rules (ALL-CAPS, '!!!') are preserved.

    Expects df to have 'text', 'cleaned', and 'vader_input' columns.
    """
    def _spacy_counts(text: str) -> Tuple[int, int, int]:
        doc = nlp(text[:1000])
        verbs = sum(1 for t in doc if t.pos_ == 'VERB')
        adjs  = sum(1 for t in doc if t.pos_ == 'ADJ')
        advs  = sum(1 for t in doc if t.pos_ == 'ADV')
        return verbs, adjs, advs

    def _negation_density(text: str) -> float:
        tokens = text.lower().split()
        if not tokens:
            return 0.0
        return sum(1 for t in tokens if t in NEGATION_WORDS) / len(tokens)

    def _ttr(text: str) -> float:
        tokens = text.lower().split()
        if not tokens:
            return 0.0
        return len(set(tokens)) / len(tokens)

    def _sentence_vader_stats(text: str) -> Tuple[float, float, float, float]:
        """
        Tokenise into sentences, score each with VADER, return:
          (std_dev_of_compounds, last_sentence_compound, contrast, first_sentence_compound)
        contrast = abs(mean_pos − mean_neg) across sentences.
        Falls back gracefully for single-sentence reviews.
        Uses vader_input text (casing/punctuation preserved).
        """
        sentences = nltk.sent_tokenize(str(text))
        if len(sentences) < 2:
            full = analyzer.polarity_scores(str(text))
            cmp  = full['compound']
            return 0.0, cmp, abs(full['pos'] - full['neg']), cmp
        scores     = [analyzer.polarity_scores(s) for s in sentences]
        compounds  = [s['compound'] for s in scores]
        std_dev    = float(np.std(compounds))
        last_cmp   = compounds[-1]
        first_cmp  = compounds[0]
        contrast   = float(abs(np.mean([s['pos'] for s in scores]) -
                                np.mean([s['neg'] for s in scores])))
        return std_dev, last_cmp, contrast, first_cmp

    df = df.copy()

    # ── VADER: scored on vader_input (HTML-clean but casing/punct intact) ──
    vader_scores = df['vader_input'].apply(
        lambda x: analyzer.polarity_scores(str(x)))
    df['vader_pos']      = vader_scores.apply(lambda s: s['pos'])
    df['vader_neu']      = vader_scores.apply(lambda s: s['neu'])
    df['vader_neg']      = vader_scores.apply(lambda s: s['neg'])
    df['vader_compound'] = vader_scores.apply(lambda s: s['compound'])

    # ── TextBlob: also on vader_input ──
    tb_scores = df['vader_input'].apply(
        lambda x: TextBlob(str(x)).sentiment)
    df['textblob_polarity']     = tb_scores.apply(lambda s: s.polarity)
    df['textblob_subjectivity'] = tb_scores.apply(lambda s: s.subjectivity)

    # ── spaCy POS counts (on cleaned — lowercased is fine for POS) ──
    pos_res = df['cleaned'].apply(_spacy_counts)
    df['verb_count'], df['adj_count'], df['adv_count'] = zip(*pos_res)

    # ── Keyword counts (on cleaned) ──
    df['pos_keyword_count'] = df['cleaned'].apply(
        lambda x: len(POS_KEYWORDS.findall(x)))
    df['neg_keyword_count'] = df['cleaned'].apply(
        lambda x: len(NEG_KEYWORDS.findall(x)))

    # ── Negation density (on cleaned) ──
    df['negation_density'] = df['cleaned'].apply(_negation_density)

    # ── Surface / structural features ──
    df['word_count']        = df['cleaned'].str.split().str.len()
    df['exclamation_count'] = df['text'].astype(str).str.count('!')
    df['question_count']    = df['text'].astype(str).str.count(r'\?')
    df['type_token_ratio']  = df['cleaned'].apply(_ttr)
    df['sentence_count']    = df['text'].astype(str).str.count(r'[.!?]+')

    # ── Sentence-level VADER stats (on vader_input — preserves emphasis) ──
    svader = df['vader_input'].apply(_sentence_vader_stats)
    df['sentence_vader_std']   = svader.apply(lambda t: t[0])
    df['last_sentence_vader']  = svader.apply(lambda t: t[1])
    df['vader_contrast']       = svader.apply(lambda t: t[2])
    df['first_sentence_vader'] = svader.apply(lambda t: t[3])

    # ── Caps ratio (on raw text — preserves original casing) ──
    def _caps_ratio(text: str) -> float:
        text = str(text)
        alpha = [c for c in text if c.isalpha()]
        if not alpha:
            return 0.0
        return sum(1 for c in alpha if c.isupper()) / len(alpha)

    df['caps_ratio']          = df['text'].astype(str).apply(_caps_ratio)
    df['avg_sentence_length'] = (df['word_count'] /
                                  df['sentence_count'].clip(lower=1))

    # ── Intent flags (on raw text — preserves apostrophes for regex) ──
    df['negative_intent_flag'] = df['text'].astype(str).apply(
        lambda x: int(bool(NEGATIVE_INTENT_RE.search(x))))
    df['positive_intent_flag'] = df['text'].astype(str).apply(
        lambda x: int(bool(POSITIVE_INTENT_RE.search(x))))

    feature_cols = [
        'vader_pos', 'vader_neu', 'vader_neg', 'vader_compound',
        'textblob_polarity', 'textblob_subjectivity',
        'verb_count', 'adj_count', 'adv_count',
        'pos_keyword_count', 'neg_keyword_count',
        'negation_density', 'word_count',
        'exclamation_count', 'question_count',
        'type_token_ratio', 'sentence_count',
        'sentence_vader_std', 'vader_contrast', 'last_sentence_vader',
        'negative_intent_flag', 'positive_intent_flag',
        'caps_ratio', 'avg_sentence_length', 'first_sentence_vader',
    ]
    return df[feature_cols].values.astype(float)


print('Preprocessing helpers defined. Meta feature count: 25')


In [ ]:
# ─────────────────────────────────────────────
# Cell 7 | Load, Clean & Split — Zero Leakage
# ─────────────────────────────────────────────
def load_and_prepare(file_path: str):
    """
    Returns
    -------
    X_tr, X_val, X_te  : scaled sparse feature matrices
    y_tr, y_val, y_te  : integer label arrays
    tfidf_obj          : fitted TfidfVectorizer (for serialisation)
    scaler_obj         : fitted MaxAbsScaler    (for serialisation)
    delta_vec          : delta weight vector    (for serialisation)
    df_tr, df_te       : raw DataFrames (for error analysis)
    """
    print('Loading data...')
    df = pd.read_csv(file_path)
    df['label'] = df['stars'].map({1: 0, 2: 0, 3: 1, 4: 2, 5: 2})
    df = df.dropna(subset=['text', 'label']).reset_index(drop=True)
    print(f'  Rows after cleaning: {len(df):,}')
    print('  Class counts:', dict(zip(*np.unique(df['label'], return_counts=True))))

    print('Cleaning text...')
    df['cleaned']     = df['text'].astype(str).apply(clean_text)
    df['vader_input'] = df['text'].astype(str).apply(make_vader_input)  # HTML-clean only; casing+punct preserved for VADER

    # ── Stratified 70 / 15 / 15 split — done BEFORE any fitting ──
    y   = df['label'].values
    idx = np.arange(len(df))

    idx_tr, idx_tmp, y_tr, y_tmp = train_test_split(
        idx, y, test_size=0.30, random_state=SEED, stratify=y)
    idx_val, idx_te, y_val, y_te = train_test_split(
        idx_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

    df_tr  = df.iloc[idx_tr].reset_index(drop=True)
    df_val = df.iloc[idx_val].reset_index(drop=True)
    df_te  = df.iloc[idx_te].reset_index(drop=True)

    print(f'Split — Train: {len(df_tr):,}  Val: {len(df_val):,}  Test: {len(df_te):,}')

    # ── TF-IDF: fit on TRAIN only, transform all splits ──
    print('Fitting TF-IDF on training set only (no leakage)...')
    tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=15000, min_df=2, sublinear_tf=True)
    X_tr_tfidf  = tfidf.fit_transform(df_tr['cleaned'])   # fit + transform
    X_val_tfidf = tfidf.transform(df_val['cleaned'])       # transform only
    X_te_tfidf  = tfidf.transform(df_te['cleaned'])        # transform only

    # ── Delta weights: 3-class delta computed from TRAIN only ──
    # pos_avg - neg_avg: discriminates positive vs negative (original)
    # neutral_boost: lifts terms that are neutral-specific vs both extremes
    # Combined delta gives the Neutral class its own discriminative signal
    pos_avg  = np.asarray(X_tr_tfidf[y_tr == 2].mean(axis=0)).flatten()
    neg_avg  = np.asarray(X_tr_tfidf[y_tr == 0].mean(axis=0)).flatten()
    neu_avg  = np.asarray(X_tr_tfidf[y_tr == 1].mean(axis=0)).flatten()
    polar_delta   = pos_avg - neg_avg                          # Neg↔Pos axis
    neutral_boost = neu_avg - (pos_avg + neg_avg) / 2.0       # Neutral vs extremes
    delta         = polar_delta + 0.5 * neutral_boost         # combined 3-class delta
    X_tr_tfidf  = X_tr_tfidf.multiply(delta)
    X_val_tfidf = X_val_tfidf.multiply(delta)
    X_te_tfidf  = X_te_tfidf.multiply(delta)

    # ── Meta features ──
    print('Extracting meta features (25 features: VADER×4, TextBlob×2, POS×3, keywords×2, negation, surface×4, sentence_vader_std, vader_contrast, last_sentence_vader, intent×2, caps_ratio, avg_sentence_length, first_sentence_vader)...')
    meta_tr  = extract_meta_features(df_tr)
    meta_val = extract_meta_features(df_val)
    meta_te  = extract_meta_features(df_te)

    X_tr_raw  = hstack([X_tr_tfidf,  csr_matrix(meta_tr)])
    X_val_raw = hstack([X_val_tfidf, csr_matrix(meta_val)])
    X_te_raw  = hstack([X_te_tfidf,  csr_matrix(meta_te)])

    # ── Scaler: fit on TRAIN only, transform all splits ──
    scaler   = MaxAbsScaler()
    X_tr_sc  = scaler.fit_transform(X_tr_raw)   # fit + transform
    X_val_sc = scaler.transform(X_val_raw)        # transform only
    X_te_sc  = scaler.transform(X_te_raw)         # transform only

    print('Preprocessing complete.')
    return (X_tr_sc, X_val_sc, X_te_sc,
            y_tr, y_val, y_te,
            tfidf, scaler, delta,
            df_tr, df_te)


(X_tr, X_val, X_te,
 y_tr, y_val, y_te,
 tfidf_obj, scaler_obj, delta_vec,
 df_train_raw, df_test_raw) = load_and_prepare(CSV_PATH)

print(f'\nFeature dimensions: {X_tr.shape[1]:,}')

---
## Section 2b — Descriptive Analytics of Engineered Features

After feature engineering we inspect the 25 meta-features to verify that they
carry real discriminative signal before handing the data to the classifier:
- **Per-class feature means** — do VADER / TextBlob / keyword features separate the classes?
- **Feature distributions** — histograms and box-plots for the most informative features.
- **Correlation heat-map** — which meta-features are redundant?
- **Train / Val / Test label balance check** — confirms stratification worked.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 7b | Descriptive Analytics — Engineered Feature Space
# ─────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

META_FEATURE_NAMES = [
    'vader_pos', 'vader_neu', 'vader_neg', 'vader_compound',
    'textblob_polarity', 'textblob_subjectivity',
    'verb_count', 'adj_count', 'adv_count',
    'pos_keyword_count', 'neg_keyword_count',
    'negation_density', 'word_count',
    'exclamation_count', 'question_count',
    'type_token_ratio', 'sentence_count',
    'sentence_vader_std', 'vader_contrast', 'last_sentence_vader',
    'negative_intent_flag', 'positive_intent_flag',
    'caps_ratio', 'avg_sentence_length', 'first_sentence_vader',
]

# Rebuild train meta-features as a DataFrame for inspection
# (extract_meta_features already ran inside load_and_prepare; we re-run
#  on df_train_raw which is already cleaned and split — no leakage)
meta_tr_arr = extract_meta_features(df_train_raw)
meta_df = pd.DataFrame(meta_tr_arr, columns=META_FEATURE_NAMES)
meta_df['label'] = y_tr
meta_df['label_name'] = meta_df['label'].map({0: 'Negative', 1: 'Neutral', 2: 'Positive'})

label_colors = {'Negative': '#e74c3c', 'Neutral': '#f39c12', 'Positive': '#2ecc71'}

# ── 1. Train / Val / Test label balance check ──────────────────────────────
print('=== Label Distribution After Stratified Split ===')
for split_name, y_sp in [('Train', y_tr), ('Val', y_val), ('Test', y_te)]:
    uniq, cnts = np.unique(y_sp, return_counts=True)
    row = {LABELS[u]: c for u, c in zip(uniq, cnts)}
    pcts = {k: f'{100*v/len(y_sp):.1f}%' for k, v in row.items()}
    print(f'  {split_name:5s}  n={len(y_sp):,}  counts={row}  pcts={pcts}')

# ── 2. Per-class mean table for top sentiment features ─────────────────────
key_features = [
    'vader_compound', 'vader_pos', 'vader_neu', 'vader_neg',
    'textblob_polarity', 'textblob_subjectivity',
    'pos_keyword_count', 'neg_keyword_count',
    'negation_density', 'exclamation_count', 'caps_ratio',
    'negative_intent_flag', 'positive_intent_flag',
    'sentence_vader_std', 'word_count',
]
print('\n=== Per-Class Feature Means (Training Set) ===')
mean_table = (meta_df.groupby('label_name')[key_features]
              .mean().T.round(4))
print(mean_table.to_string())

# ── 3. Feature distribution plots — 8 most informative features ───────────
plot_features = [
    'vader_compound', 'textblob_polarity', 'vader_neu',
    'pos_keyword_count', 'neg_keyword_count',
    'negation_density', 'word_count', 'sentence_vader_std',
]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, feat in zip(axes, plot_features):
    for lname, color in label_colors.items():
        vals = meta_df[meta_df['label_name'] == lname][feat]
        # Clip outliers for visibility
        p99 = vals.quantile(0.99)
        vals_clipped = vals.clip(upper=p99)
        ax.hist(vals_clipped, bins=40, alpha=0.50, color=color,
                label=lname, density=True)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('Value', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6)
fig.suptitle('Meta-Feature Distributions by Class (Training Set, p99 clip)', fontsize=13)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.show()

# ── 4. Box-plots for key VADER / TextBlob features ────────────────────────
box_features = ['vader_compound', 'vader_pos', 'vader_neu', 'vader_neg',
                'textblob_polarity', 'textblob_subjectivity']
fig, axes = plt.subplots(1, len(box_features), figsize=(18, 4))
order = ['Negative', 'Neutral', 'Positive']
colors = [label_colors[l] for l in order]
for ax, feat in zip(axes, box_features):
    data_by_class = [meta_df[meta_df['label_name'] == l][feat].values for l in order]
    bp = ax.boxplot(data_by_class, patch_artist=True, labels=order,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(axis='x', rotation=20, labelsize=7)
fig.suptitle('VADER & TextBlob Feature Box-Plots by Class (Training Set)', fontsize=12)
plt.tight_layout()
plt.savefig('feature_boxplots.png', bbox_inches='tight')
plt.show()

# ── 5. Meta-feature correlation heat-map ──────────────────────────────────
corr = meta_df[META_FEATURE_NAMES].corr()
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, mask=mask, annot=False, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Meta-Feature Correlation Matrix (Training Set)', fontsize=12)
ax.tick_params(labelsize=7)
plt.tight_layout()
plt.savefig('feature_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# ── 6. Intent-flag and binary feature summary ─────────────────────────────
binary_features = ['negative_intent_flag', 'positive_intent_flag']
print('\n=== Binary Feature Hit Rates by Class (Training Set) ===')
for feat in binary_features:
    row = meta_df.groupby('label_name')[feat].mean().round(4)
    print(f'  {feat}: {row.to_dict()}')

print('\nDescriptive analytics of engineered features complete.')


---
## Section 3 — Evaluation Helper

All models funnel through a single `evaluate()` function that prints:
- Overall **Accuracy** and **Macro-F1**
- Per-class **Precision**, **Recall**, and **F1** for Neg / Neu / Pos
- A **confusion matrix** heatmap
- Appends a full row to `results_log` for the final comparison table

In [ ]:
# ─────────────────────────────────────────────
# Cell 8 | Central Evaluation Function
# ─────────────────────────────────────────────
def evaluate(model_or_preds, X_test, y_test: np.ndarray,
             name: str, precomputed: bool = False) -> np.ndarray:
    """
    Evaluate a trained model or pre-computed predictions on one split.

    Parameters
    ----------
    model_or_preds : fitted model with .predict(), or np.ndarray if precomputed=True
    X_test         : feature matrix (ignored when precomputed=True)
    y_test         : true labels
    name           : display name for the model
    precomputed    : set True when passing a predictions array directly

    Returns
    -------
    y_pred : np.ndarray of predicted labels
    """
    y_pred = model_or_preds if precomputed else model_or_preds.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    mf1 = f1_score(y_test, y_pred, average='macro')
    rpt = classification_report(
        y_test, y_pred, target_names=LABELS, output_dict=True, zero_division=0)

    # ── Console output ──
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  MODEL : {name}')
    print(sep)
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Macro-F1  : {mf1:.4f}')
    print()
    per_class = pd.DataFrame(rpt).T.loc[LABELS, ['precision', 'recall', 'f1-score', 'support']]
    per_class.columns = ['Precision', 'Recall', 'F1', 'Support']
    per_class['Support'] = per_class['Support'].astype(int)
    print(per_class.round(4).to_string())
    print(sep)

    # ── Log to results table ──
    results_log.append({
        'Model':         name,
        'Accuracy':      round(acc, 4),
        'Macro F1':      round(mf1, 4),
        'Neg Precision': round(rpt['Neg']['precision'],  4),
        'Neg Recall':    round(rpt['Neg']['recall'],     4),
        'Neg F1':        round(rpt['Neg']['f1-score'],   4),
        'Neu Precision': round(rpt['Neu']['precision'],  4),
        'Neu Recall':    round(rpt['Neu']['recall'],     4),
        'Neu F1':        round(rpt['Neu']['f1-score'],   4),
        'Pos Precision': round(rpt['Pos']['precision'],  4),
        'Pos Recall':    round(rpt['Pos']['recall'],     4),
        'Pos F1':        round(rpt['Pos']['f1-score'],   4),
    })

    # ── Confusion matrix ──
    cm  = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS).plot(
        cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}')
    plt.tight_layout()
    safe_name = name.replace(' ', '_').replace('/', '-').replace('(', '').replace(')', '')
    plt.savefig(f'cm_{safe_name}.png', bbox_inches='tight')
    plt.show()

    return y_pred


def evaluate_all_splits(model, X_tr, y_tr, X_val, y_val, X_te, y_te,
                         model_name: str) -> np.ndarray:
    """
    Run evaluate() on Train, Validation, and Test splits and print a
    consolidated three-split summary table.  Only the Test split row is
    appended to results_log (to keep the final comparison table clean).

    Returns
    -------
    y_pred_test : predictions on the test set
    """
    split_rows = []
    for split_name, X_sp, y_sp in [
            ('Train', X_tr, y_tr),
            ('Val',   X_val, y_val),
            ('Test',  X_te,  y_te)]:
        y_p = model.predict(X_sp)
        acc = accuracy_score(y_sp, y_p)
        mf1 = f1_score(y_sp, y_p, average='macro')
        rpt = classification_report(
            y_sp, y_p, target_names=LABELS, output_dict=True, zero_division=0)
        split_rows.append({
            'Split':       split_name,
            'Accuracy':    round(acc, 4),
            'Macro F1':    round(mf1, 4),
            'Neg F1':      round(rpt['Neg']['f1-score'], 4),
            'Neu F1':      round(rpt['Neu']['f1-score'], 4),
            'Pos F1':      round(rpt['Pos']['f1-score'], 4),
        })

    print(f'\n{"=" * 65}')
    print(f'  TRAIN / VAL / TEST SUMMARY — {model_name}')
    print(f'{"=" * 65}')
    print(pd.DataFrame(split_rows).set_index('Split').to_string())
    print(f'{"=" * 65}')

    # Append only test-split row to global log
    y_pred_test = model.predict(X_te)
    evaluate(model, X_te, y_te, model_name)
    return y_pred_test


print('evaluate() and evaluate_all_splits() defined.')


---
## Section 4 — Baselines

Two trivial baselines establish the minimum bar that any learned model must surpass:
1. **Majority-class classifier** — always predicts the most frequent class.
2. **VADER rule-based** — compound ≥ 0.05 → Positive, ≤ −0.05 → Negative, else Neutral. No training required.

A model that cannot beat these on Macro-F1 is not worth deploying.

In [ ]:
# ─────────────────────────────────────────────
# Cell 9 | Baselines
# ─────────────────────────────────────────────

# Baseline 1 — Majority-class
dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy.fit(X_tr, y_tr)
evaluate(dummy, X_te, y_te, 'Majority-Class Baseline')

# Baseline 2 — VADER rule-based (no training needed)
def vader_predict(texts: np.ndarray) -> np.ndarray:
    preds = []
    for t in texts:
        score = analyzer.polarity_scores(str(t))['compound']
        preds.append(2 if score >= 0.05 else (0 if score <= -0.05 else 1))
    return np.array(preds)

vader_preds = vader_predict(df_test_raw['text'].values)
evaluate(vader_preds, None, y_te, 'VADER Rule-Based Baseline', precomputed=True)

---
## Section 5 — Linear SVC with Optuna Tuning

Three stages:
1. **Untuned** Linear SVC with `class_weight='balanced'` — first learned model reference point.
2. **Optuna tuning** — 5-fold stratified CV over the training set, expanded search space for `C` and `loss`, optimises for Macro-F1. 40 trials.
3. **Tuned model** retrained on the full training set with the best hyperparameters.

### Tuning improvements over baseline
- Search space expanded: `C` range widened to `(1e-3, 100)`, `loss` (`hinge` vs `squared_hinge`) included
- Trial count raised from 20 → 40 for more thorough exploration
- 5-fold stratified CV ensures Neutral class is proportionally represented in every fold
- `TPESampler` with fixed seed for full reproducibility


In [ ]:
# ─────────────────────────────────────────────
# Cell 10 | Linear SVC Suite (Fixed Tuning)
# ─────────────────────────────────────────────
def run_linear_svc_suite(X_tr, y_tr, X_val, y_val, X_te, y_te):
    print('\n' + '=' * 55)
    print('LINEAR SVC SUITE')
    print('=' * 55)

    # ── 1. Untuned baseline ──
    print('\n[1/3] Training untuned Linear SVC...')
    base_lsvc = LinearSVC(
        dual=False, class_weight='balanced',
        random_state=SEED, max_iter=5000)
    base_lsvc.fit(X_tr, y_tr)
    evaluate_all_splits(base_lsvc, X_tr, y_tr, X_val, y_val, X_te, y_te,
                        'Linear SVC — Untuned')

    # ── 2. Optuna: 5-fold CV on X_tr only ──
    # KEY FIX: cross_val_score is called on X_tr (the same data size the
    # final model trains on).  Previously the study's best C was found via
    # CV on X_tr but the final model was sometimes retrained on X_tr+X_val,
    # causing a data-size mismatch in effective regularisation — the larger
    # dataset needs a lower C to achieve equivalent regularisation strength,
    # so the tuned model was under-regularised and scored *worse* than the
    # untuned one.  Keeping training data fixed to X_tr throughout fixes this.
    print('\n[2/3] Optuna tuning — 5-fold CV on X_tr, 60 trials...')
    cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    def objective(trial):
        C    = trial.suggest_float('C', 1e-3, 500.0, log=True)
        loss = trial.suggest_categorical('loss', ['hinge', 'squared_hinge'])
        dual = (loss == 'hinge')   # hinge requires dual=True
        model = LinearSVC(
            C=C, loss=loss, dual=dual,
            class_weight='balanced',
            random_state=SEED, max_iter=5000)
        # Evaluate *only* on X_tr — no val-set leakage into hyperparameter search
        scores = cross_val_score(
            model, X_tr, y_tr,
            cv=cv_splitter, scoring='f1_macro', n_jobs=-1)
        return scores.mean()

    sampler    = optuna.samplers.TPESampler(seed=SEED)
    study_lsvc = optuna.create_study(direction='maximize', sampler=sampler)
    study_lsvc.optimize(objective, n_trials=60)

    best      = study_lsvc.best_params
    best_C    = best['C']
    best_loss = best['loss']
    best_dual = (best_loss == 'hinge')
    print(f'  Best C         = {best_C:.5f}')
    print(f'  Best loss      = {best_loss}')
    print(f'  CV Macro-F1    = {study_lsvc.best_value:.4f}')

    # ── Optuna trial history plot ──
    trial_vals   = [t.value for t in study_lsvc.trials if t.value is not None]
    running_best = [max(trial_vals[:i+1]) for i in range(len(trial_vals))]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(trial_vals,   alpha=0.4, label='Trial Macro-F1', color='#3498db')
    ax.plot(running_best, color='#e74c3c', linewidth=2, label='Running Best')
    ax.set_xlabel('Trial')
    ax.set_ylabel('CV Macro-F1')
    ax.set_title('Optuna Trial History — Linear SVC')
    ax.legend()
    plt.tight_layout()
    plt.savefig('optuna_trial_history.png', bbox_inches='tight')
    plt.show()

    # ── 3. Final tuned model — trained on X_tr (same size as CV folds) ──
    # IMPORTANT: do NOT train on X_tr+X_val here.  The best C found by Optuna
    # was calibrated for a dataset of len(X_tr) samples.  Expanding the training
    # set to len(X_tr)+len(X_val) changes the effective regularisation, which
    # can make the tuned model perform *worse* than the untuned one — the bug
    # this fix addresses.
    print('\n[3/3] Training final tuned Linear SVC on X_tr (matches CV data size)...')
    tuned_lsvc = LinearSVC(
        C=best_C, loss=best_loss, dual=best_dual,
        class_weight='balanced',
        random_state=SEED, max_iter=5000)
    tuned_lsvc.fit(X_tr, y_tr)

    preds_tuned = evaluate_all_splits(
        tuned_lsvc, X_tr, y_tr, X_val, y_val, X_te, y_te,
        'Linear SVC — Tuned')

    return base_lsvc, tuned_lsvc, preds_tuned, study_lsvc


base_lsvc, tuned_lsvc, lsvc_preds, lsvc_study = run_linear_svc_suite(
    X_tr, y_tr, X_val, y_val, X_te, y_te)


---
## Section 6 — Error Analysis

We inspect misclassified test samples to understand where the best model falls short.
Focus areas:
- **Neutral class errors** — these are hardest; reviews near the Neg/Pos boundary.
- **Confusion breakdown** — which class pairs confuse the model most?
- **Sample texts** — concrete examples with VADER scores for qualitative insight.


In [ ]:
# ─────────────────────────────────────────────
# Cell 12 | Error Analysis
# ─────────────────────────────────────────────
def error_analysis(y_true: np.ndarray, y_pred: np.ndarray,
                   df_raw: pd.DataFrame,
                   model_name: str = 'Model', n_samples: int = 5) -> None:
    """
    Print a structured breakdown of misclassifications.
    df_raw must be aligned with y_true (same row order).
    """
    assert len(y_true) == len(df_raw), 'y_true and df_raw must have the same length'
    lmap = {0: 'Neg', 1: 'Neu', 2: 'Pos'}

    err = df_raw.copy().reset_index(drop=True)
    err['y_true'] = y_true
    err['y_pred'] = y_pred
    err['vader']  = err['text'].astype(str).apply(
        lambda x: analyzer.polarity_scores(x)['compound'])
    err = err[err['y_true'] != err['y_pred']].reset_index(drop=True)

    print(f'\n{"=" * 60}')
    print(f'  ERROR ANALYSIS — {model_name}')
    print(f'{"=" * 60}')
    print(f'  Total misclassified: {len(err):,} / {len(y_true):,} '
          f'({100 * len(err) / len(y_true):.1f}%)')

    # Confusion breakdown
    print('\n  Confusion breakdown (True → Predicted):')
    bd = (err.groupby(['y_true', 'y_pred']).size()
          .reset_index(name='count')
          .sort_values('count', ascending=False))
    bd['True']      = bd['y_true'].map(lmap)
    bd['Predicted'] = bd['y_pred'].map(lmap)
    print(bd[['True', 'Predicted', 'count']].to_string(index=False))

    # Neutral errors (hardest class)
    print('\n  Neutral class error detail:')
    neu_err = err[err['y_true'] == 1]
    total_neu = int((y_true == 1).sum())
    print(f'  Neutral misclassified: {len(neu_err):,} / {total_neu:,} '
          f'({100 * len(neu_err) / max(1, total_neu):.1f}%)')
    for _, row in neu_err.sample(
            min(n_samples, len(neu_err)), random_state=SEED).iterrows():
        print(f'\n    True: Neu → Pred: {lmap[row["y_pred"]]} '
              f'| VADER: {row["vader"]:.3f}')
        print(f'    "{str(row["text"])[:200]}"')

    # Other-class errors (sample)
    print('\n  Other misclassifications (sample):')
    other_err = err[err['y_true'] != 1]
    for _, row in other_err.sample(
            min(n_samples, len(other_err)), random_state=SEED).iterrows():
        print(f'\n    True: {lmap[row["y_true"]]} → Pred: {lmap[row["y_pred"]]} '
              f'| VADER: {row["vader"]:.3f}')
        print(f'    "{str(row["text"])[:200]}"')


# Run on the tuned Linear SVC (fastest model, use for qualitative analysis)
error_analysis(y_te, lsvc_preds, df_test_raw, 'Linear SVC — Tuned')

---
## Section 7 — Performance Summary

Aggregated comparison of all models — baselines and learned classifiers — showing Accuracy, Macro-F1, and full per-class Precision / Recall / F1.


In [ ]:
# ─────────────────────────────────────────────
# Cell 13 | Final Performance Summary
# ─────────────────────────────────────────────
print('\n' + '=' * 100)
print('FINAL PERFORMANCE SUMMARY — ALL MODELS (Test Set)')
print('=' * 100)

summary = pd.DataFrame(results_log)
# Display in two parts so it fits cleanly
overview_cols  = ['Model', 'Accuracy', 'Macro F1']
perclass_cols  = ['Model',
                  'Neg Precision', 'Neg Recall', 'Neg F1',
                  'Neu Precision', 'Neu Recall', 'Neu F1',
                  'Pos Precision', 'Pos Recall', 'Pos F1']

print('\n--- Overview ---')
print(summary[overview_cols].to_string(index=False))
print('\n--- Per-Class Metrics ---')
print(summary[perclass_cols].to_string(index=False))

# Bar chart — Macro-F1
palette = ['#95a5a6', '#7f8c8d', '#3498db', '#2980b9', '#e74c3c']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].barh(summary['Model'], summary['Macro F1'],
                    color=palette[:len(summary)], edgecolor='white')
axes[0].set_xlabel('Macro F1')
axes[0].set_title('Macro-F1 by Model')
axes[0].set_xlim(0, 1.05)
for bar, val in zip(bars, summary['Macro F1']):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', fontsize=8)

# Grouped per-class F1 (learned models only)
learned = summary[~summary['Model'].str.contains('Baseline')].reset_index(drop=True)
x = np.arange(len(learned))
w = 0.25
axes[1].bar(x - w,  learned['Neg F1'], width=w, label='Neg F1', color='#e74c3c')
axes[1].bar(x,      learned['Neu F1'], width=w, label='Neu F1', color='#f39c12')
axes[1].bar(x + w,  learned['Pos F1'], width=w, label='Pos F1', color='#2ecc71')
axes[1].set_xticks(x)
axes[1].set_xticklabels(learned['Model'], rotation=15, ha='right', fontsize=8)
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Per-Class F1 — Learned Models')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

---
## Section 8 — Model Serialization

Tuned model saved alongside the shared preprocessing pipeline.

| File | Contents |
|------|----------|
| `preprocessing_pipeline.pkl` | `tfidf`, `delta`, `scaler` — needed for inference on new text |
| `model_lsvc_untuned.pkl` | Untuned Linear SVC |
| `model_lsvc_tuned.pkl` | Optuna-tuned Linear SVC (best `C` and `loss`) |
| `full_bundle.pkl` | All of the above in one dict (convenience) |

A smoke-test confirms end-to-end inference works after reloading.


In [ ]:
# ─────────────────────────────────────────────
# Cell 13 | Save Tuned Models + Pipeline
# ─────────────────────────────────────────────

# 1. Shared preprocessing artifacts
preprocessing = {
    'tfidf':  tfidf_obj,
    'delta':  delta_vec,
    'scaler': scaler_obj,
    'labels': LABELS,
    'seed':   SEED,
}
joblib.dump(preprocessing, 'preprocessing_pipeline.pkl')
print('Saved: preprocessing_pipeline.pkl')

# 2. Individual model files
joblib.dump(base_lsvc,  'model_lsvc_untuned.pkl')
print('Saved: model_lsvc_untuned.pkl')

joblib.dump(tuned_lsvc, 'model_lsvc_tuned.pkl')
print('Saved: model_lsvc_tuned.pkl')

# 3. Full convenience bundle
full_bundle = {
    **preprocessing,
    'lsvc_untuned': base_lsvc,
    'lsvc_tuned':   tuned_lsvc,
}
joblib.dump(full_bundle, 'full_bundle.pkl')
print('Saved: full_bundle.pkl')

print('\n✓ All models and preprocessing pipeline saved.')


In [ ]:
# ─────────────────────────────────────────────
# Cell 14 | Smoke-Test: Reload & Infer on New Text
# ─────────────────────────────────────────────
def predict_new(texts: List[str], bundle: dict,
                model_key: str = 'lsvc_tuned') -> List[str]:
    """
    End-to-end inference on raw text strings using a saved bundle.

    Parameters
    ----------
    texts     : list of raw review strings
    bundle    : loaded full_bundle dict
    model_key : one of 'lsvc_untuned', 'lsvc_tuned'

    Returns
    -------
    list of label strings ('Neg', 'Neu', 'Pos')
    """
    cleaned     = [clean_text(t) for t in texts]
    vader_texts = [make_vader_input(t) for t in texts]   # HTML-clean, casing intact
    X_tfidf = bundle['tfidf'].transform(cleaned).multiply(bundle['delta'])

    meta_rows = []
    for raw_t, cl_t, vt in zip(texts, cleaned, vader_texts):
        # VADER & TextBlob on vader_input (preserves casing/punctuation)
        vs    = analyzer.polarity_scores(vt)
        tb    = TextBlob(vt).sentiment
        wc    = len(str(raw_t).split())
        neg_d = sum(1 for w in cl_t.split() if w in NEGATION_WORDS) / max(1, len(cl_t.split()))
        ttr   = len(set(cl_t.split())) / max(1, len(cl_t.split()))

        # Sentence-level VADER stats (on vader_input)
        _sents = nltk.sent_tokenize(vt)
        if len(_sents) >= 2:
            _svs       = [analyzer.polarity_scores(s) for s in _sents]
            _cmps      = [s['compound'] for s in _svs]
            _sv_std    = float(np.std(_cmps))
            _last_cmp  = _cmps[-1]
            _first_cmp = _cmps[0]
            _contrast  = float(abs(np.mean([s['pos'] for s in _svs]) -
                                   np.mean([s['neg'] for s in _svs])))
        else:
            _sv_std   = 0.0
            _last_cmp = _first_cmp = vs['compound']
            _contrast = abs(vs['pos'] - vs['neg'])

        meta_rows.append([
            vs['pos'], vs['neu'], vs['neg'], vs['compound'],   # VADER ×4
            tb.polarity, tb.subjectivity,                       # TextBlob ×2
            0, 0, 0,                                            # verb/adj/adv (0 for speed)
            len(POS_KEYWORDS.findall(cl_t)),                    # pos keywords (on cleaned)
            len(NEG_KEYWORDS.findall(cl_t)),                    # neg keywords (on cleaned)
            neg_d,                                              # negation density
            wc,                                                 # word count
            str(raw_t).count('!'),                              # exclamations
            str(raw_t).count('?'),                              # questions
            ttr,                                                # type-token ratio
            str(raw_t).count('.') + str(raw_t).count('!') + str(raw_t).count('?'),  # sentences
            _sv_std,                                            # sentence_vader_std
            _contrast,                                          # vader_contrast
            _last_cmp,                                          # last_sentence_vader
            int(bool(NEGATIVE_INTENT_RE.search(str(raw_t)))),  # negative_intent_flag
            int(bool(POSITIVE_INTENT_RE.search(str(raw_t)))),  # positive_intent_flag
            # NEW features (#4)
            sum(1 for c in [x for x in str(raw_t) if x.isalpha()] if c.isupper()) /
                max(1, sum(1 for c in str(raw_t) if c.isalpha())),  # caps_ratio
            wc / max(1, str(raw_t).count('.') + str(raw_t).count('!') +
                     str(raw_t).count('?')),                        # avg_sentence_length
            _first_cmp,                                             # first_sentence_vader
        ])

    X_meta = csr_matrix(np.array(meta_rows, dtype=float))
    X_all  = bundle['scaler'].transform(hstack([X_tfidf, X_meta]))
    preds  = bundle[model_key].predict(X_all)
    return [bundle['labels'][int(p)] for p in preds]


# Reload from disk to confirm serialisation works end-to-end
loaded_bundle = joblib.load('full_bundle.pkl')

test_texts = [
    'This recipe was absolutely amazing, best I have ever tasted!',
    'It was okay, nothing special but not bad either.',
    'Terrible! Completely inedible, wasted all my ingredients.',
    'Pretty good overall, though a bit too salty for my taste.',
    'Not bad, not great — pretty average experience.',
    'I would not make this again, way too bland and dry.',
]

print('Smoke-test — inference with saved models:')
print(f'{"Text":<57}  {"Untuned":>8}  {"Tuned":>6}')
print('-' * 75)
preds_un = predict_new(test_texts, loaded_bundle, 'lsvc_untuned')
preds_tu = predict_new(test_texts, loaded_bundle, 'lsvc_tuned')
for txt, un, tu in zip(test_texts, preds_un, preds_tu):
    print(f'{txt[:55]:<57}  {un:>8}  {tu:>6}')

print('\n✓ Inference pipeline verified for both models.')


---
## Conclusions

| Change | What was done |
|--------|---------------|
| Hierarchical RBF removed | Only Linear SVC is used — better suited for sparse TF-IDF space |
| Expanded meta features (17) | Added VADER pos/neu/neg separately, TextBlob subjectivity, adverb count, negation density, exclamation/question counts, type-token ratio, sentence count, full pos+neg keyword lexicons |
| VADER neu score | Directly captures neutral-language proportion — key for Neutral class F1 |
| spaCy cap 1000 chars | Restored from v1 for better POS coverage on longer reviews |
| Improved Optuna tuning | 40 trials (vs 20), expanded C range (1e-3 to 100), loss function (`hinge` vs `squared_hinge`) also tuned, 5-fold CV per trial |
| Full per-class metrics | Every model reports Precision, Recall, F1 for Neg / Neu / Pos |
| Modular serialisation | Preprocessing pipeline and models saved separately + full bundle |
| No data leakage | TF-IDF, delta, and scaler all fit on training split only |


| 3-class delta TF-IDF | `delta = (pos−neg) + 0.5×(neu−mean(pos,neg))` — Neutral class gets discriminative TF-IDF signal |
| TF-IDF max_features 15k + min_df=2 | More trigrams captured; hapax legomena filtered out |
| Neutral class_weight tuning | Optuna searches `neutral_weight ∈ [1, 4]` instead of hardcoded `balanced` |
| C range widened to 500, 60 trials | Better coverage of high-C regime beneficial for sparse high-dim SVMs |
| Final model trained on X_tr+X_val | ~85% of data used for final fit after hyperparameter search on X_tr |
| 3 new meta features | `caps_ratio`, `avg_sentence_length`, `first_sentence_vader` |

**Recommended next steps**: Experiment with a transformer-based encoder (e.g. DistilBERT) for richer contextual representations, or add windowed negation marking in the TF-IDF vocabulary (appending `_NEG` suffix to tokens within 3 words of a negation trigger).
